In [14]:
# pip install xgboost lightgbm --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

## importing dataset

In [15]:
file_path = "/Users/meghna/Desktop/sih-2023-it-log-master/ml/coherent_access_auth_dataset_with_risk.csv"
df = pd.read_csv(file_path)
print(df.head(10))

                                  LogID  UserID       LoginTimestamp  \
0  fdaf345b-74f3-40e1-ad9b-99fe38510770  U57248  2025-02-01 04:31:35   
1  19ddde6b-8f24-4eb4-91f9-0e173e13e194  U97892  2025-01-16 11:47:43   
2  59adcf8a-a19a-4582-861a-ed2ce1ce0adb  U47441  2025-01-20 16:51:45   
3  d17bb642-fc44-4d33-b7b2-2310013240c8  U76229  2025-01-14 11:07:22   
4  e50293e0-802e-4cc9-8eed-595dc463412c  U40314  2025-01-07 16:58:44   
5  586dbc50-0ca9-4471-b543-a8bbe3dbdd26  U97205  2025-01-07 10:24:21   
6  33a68fa3-5e37-4892-9092-ade5a26b78f5  U34734  2025-01-04 02:22:59   
7  ee3f0a8c-e49c-4771-9bfa-4fb15884b862  U97464  2025-01-18 20:14:15   
8  bac5b462-77c7-4727-9fb3-3ea4de78acd2  U72759  2025-01-27 23:33:00   
9  3b1891c0-99b0-408c-a432-1869ce92ca01  U96937  2025-02-05 18:44:45   

   FailedLoginAttempts DeviceType  OS_BrowserInfo        IPAddress  \
0                    0         PC         iOS-App     36.161.195.1   
1                    3     Tablet         iOS-App     35.32.239.43 

## dropping redundant columns

In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Assuming df is the new dataset
# Handle missing values if any
df = df.dropna()


# Encoding categorical features using LabelEncoder
label_cols = ['DeviceType', 'OS_BrowserInfo', 'MFAStatus', 'APIAccess', 'PrivilegedAccess']
encoder = LabelEncoder()

for col in label_cols:
    df[col] = encoder.fit_transform(df[col])

# Extract features from LoginTimestamp
df['LoginTimestamp'] = pd.to_datetime(df['LoginTimestamp'])
df['LoginHour'] = df['LoginTimestamp'].dt.hour
df['LoginDay'] = df['LoginTimestamp'].dt.day
df['LoginWeekday'] = df['LoginTimestamp'].dt.weekday
df['LoginMonth'] = df['LoginTimestamp'].dt.month
df['LoginYear'] = df['LoginTimestamp'].dt.year

# Scaling the continuous numerical columns
scaler = MinMaxScaler()
# Adjusted to scale only the columns that exist in your dataset
continuous_cols = ['SessionDuration']  
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

# Drop columns that are not needed anymore (except UserID and Geolocation)
df.drop(columns=['LogID', 'UserID', 'LoginTimestamp', 'IPAddress', 'Geolocation'], inplace=True)

# Checking the preprocessed data
print(df.head(20))

    FailedLoginAttempts  DeviceType  OS_BrowserInfo  MFAStatus  \
0                     0           2               4          1   
1                     3           4               4          0   
2                     0           0               3          0   
3                     0           3               1          0   
4                     2           1               2          0   
5                     1           2               2          0   
6                     4           4               1          0   
7                     0           4               3          1   
8                     3           0               3          0   
9                     5           4               2          0   
10                    0           1               1          0   
11                    3           3               4          0   
12                    0           2               1          1   
13                    0           2               3          1   
14        

## label encoding Source and Destination IPs (safe and malicious)

In [17]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['RiskLevel'])
y = df['RiskLevel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set: X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Testing Set: X_test: {X_test.shape}, y_test: {y_test.shape}")


Training Set: X_train: (400, 12), y_train: (400,)
Testing Set: X_test: (100, 12), y_test: (100,)


In [18]:
base_models = [
    ('linear', LinearRegression()),
    ('random_forest', RandomForestRegressor(n_estimators=200, random_state=42)),
    ('xgboost', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
    ('lightgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
]

# Stacking model using GradientBoostingRegressor as final estimator
stacking_model = StackingRegressor(estimators=base_models, final_estimator=GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))

# 8. Train the Stacking Model
print("\nTraining the Stacking Model...")
stacking_model.fit(X_train, y_train)

# 9. Predict on the Test Data
y_pred = stacking_model.predict(X_test)

# 10. Evaluate Model Performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Model Performance:")
print(f"MSE = {mse:.4f}")
print(f"R² Score = {r2:.4f}")


Training the Stacking Model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000299 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 223
[LightGBM] [Info] Number of data points in the train set: 400, number of used features: 11
[LightGBM] [Info] Start training from score 0.561375
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [19]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

# Define parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 9],
    'subsample': [0.8, 1.0]
}

# Initialize XGBoost model
xgboost_model = XGBRegressor(random_state=42)

# Grid search for hyperparameter tuning
grid_search = GridSearchCV(estimator=xgboost_model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3)
grid_search.fit(X_train, y_train)

# Best model and predictions
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Best XGBoost Model Performance:")
print(f"MSE = {mse:.4f}")
print(f"R² Score = {r2:.4f}")

Best XGBoost Model Performance:
MSE = 0.0222
R² Score = 0.8144


## importing catboost regressor

In [20]:
from catboost import CatBoostRegressor
cat_model = CatBoostRegressor(verbose = False)

In [24]:
joblib.dump(stacking_model, "auth_fraud_model.pkl")
joblib.dump(encoder, "auth_encoder.pkl")
joblib.dump(scaler, "auth_scaler.pkl")

['auth_scaler.pkl']

## importing lightgbm regressor

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

# Example of raw input data (replace with your actual input)
raw_input_data = {
    'LogID': ['fdaf345b-74f3-40e1-ad9b-99fe38510770'],
    'UserID': ['U57248'],
    'LoginTimestamp': ['2025-02-01 04:31:35'],
    'FailedLoginAttempts': [0],
    'DeviceType': ['PC'],
    'OS_BrowserInfo': ['iOS-App'],
    'IPAddress': ['36.161.195.1'],
    'Geolocation': ['Danielleville'],
    'MFAStatus': ['Enabled'],
    'SessionDuration': [29.34],
    'APIAccess': ['Yes'],
    'PrivilegedAccess': ['Guest']
}

# Convert to DataFrame
raw_input_df = pd.DataFrame(raw_input_data)

# Preprocess the raw input (same steps as for training data)
# 1. Encoding categorical features
label_cols = ['DeviceType', 'OS_BrowserInfo', 'MFAStatus', 'APIAccess', 'PrivilegedAccess']
encoder = LabelEncoder()

for col in label_cols:
    raw_input_df[col] = encoder.fit_transform(raw_input_df[col])
    
    
    # 2. Extract features from LoginTimestamp
raw_input_df['LoginTimestamp'] = pd.to_datetime(raw_input_df['LoginTimestamp'])
raw_input_df['LoginHour'] = raw_input_df['LoginTimestamp'].dt.hour
raw_input_df['LoginDay'] = raw_input_df['LoginTimestamp'].dt.day
raw_input_df['LoginWeekday'] = raw_input_df['LoginTimestamp'].dt.weekday
raw_input_df['LoginMonth'] = raw_input_df['LoginTimestamp'].dt.month
raw_input_df['LoginYear'] = raw_input_df['LoginTimestamp'].dt.year

# 3. Scaling continuous features (SessionDuration)
scaler = MinMaxScaler()
raw_input_df['SessionDuration'] = scaler.fit_transform(raw_input_df[['SessionDuration']])

# 4. Drop unnecessary columns (LogID, UserID, IPAddress, Geolocation)
raw_input_df.drop(columns=['LogID', 'UserID', 'IPAddress', 'Geolocation'], inplace=True)

# 5. Make sure the columns match the training data columns
raw_input_df = raw_input_df[X_train.columns]

# Make the prediction using the best model from GridSearchCV
y_pred = best_model.predict(raw_input_df)

# Print the predicted RiskLevel
print(f"Predicted RiskLevel: {y_pred[0]:.4f}")


Predicted RiskLevel: 0.7668
